In [41]:
import progtv
from datetime import datetime
from pathlib import Path
import os
import glob
from datetime import datetime
from datetime import datetime, timedelta
import pandas as pd
import ollama
import re

In [2]:
tv_program = progtv.TVProgram()
current_directory = Path.cwd()
file_name_rated = f"{current_directory}/{tv_program.download_folder}/progtv_rated_{datetime.now().today().strftime('%Y-%m-%d')}.pkl"
rated_progs = tv_program.read_programs(file_name_rated)

In [10]:
rated_progs

,id,name,icon,programs
0,TF1.fr,TF1,https://www.teleboy.ch/assets/stations/308/ico...,...
1,France2.fr,France 2,https://www.teleboy.ch/assets/stations/342/ico...,name ...
2,France3.fr,France 3,https://www.teleboy.ch/assets/stations/58/icon...,name ...
3,CanalPlus.fr,Canal+,https://focus.telerama.fr/500x500/0000/00/01/c...,name ...
4,France5.fr,France 5,https://www.teleboy.ch/assets/stations/80/icon...,name sta...
5,M6.fr,M6,https://www.teleboy.ch/assets/stations/312/ico...,name ...
6,Arte.fr,Arte,https://www.teleboy.ch/assets/stations/330/ico...,...
7,C8.fr,C8,https://focus.telerama.fr/500x500/0000/00/01/c...,name start...
8,W9.fr,W9,https://www.teleboy.ch/assets/stations/268/ico...,...
9,TMC.fr,TMC,https://www.teleboy.ch/assets/stations/383/ico...,n...


In [12]:
for i, row in rated_progs.iterrows():
    print(f'min: {row["programs"]["start"].min()}, max: {row["programs"]["start"].max()}')

min: 2025-03-18 02:45:00, max: 2025-03-26 23:40:00
min: 2025-03-18 04:55:00, max: 2025-03-26 23:40:00
min: 2025-03-18 04:40:00, max: 2025-03-27 00:00:00
min: 2025-03-18 04:39:00, max: 2025-03-27 00:00:00
min: 2025-03-18 04:38:00, max: 2025-03-26 23:50:00
min: 2025-03-18 03:25:00, max: 2025-03-26 02:45:00
min: 2025-03-18 04:31:00, max: 2025-03-26 23:25:00
min: 2025-03-18 01:00:00, max: 2025-03-27 00:00:00
min: 2025-03-18 01:50:00, max: 2025-03-26 02:50:00
min: 2025-03-18 02:13:00, max: 2025-03-26 23:15:00
min: 2025-03-18 01:37:00, max: 2025-03-26 01:22:00
min: 2025-03-18 01:00:00, max: 2025-03-26 20:00:00
min: 2025-03-18 04:30:00, max: 2025-03-26 04:30:00
min: 2025-03-18 04:35:00, max: 2025-03-26 23:10:00
min: 2025-03-18 02:10:00, max: 2025-03-26 23:35:00
min: 2025-03-18 01:15:00, max: 2025-03-26 23:15:00
min: 2025-03-18 05:00:00, max: 2025-03-26 23:17:00
min: 2025-03-18 02:50:00, max: 2025-03-26 23:00:00
min: 2025-03-18 01:20:00, max: 2025-03-26 00:35:00
min: 2025-03-18 00:48:00, max: 

In [43]:
def get_sorted_files(directory):
    # Get the list of all files in the directory
    files = glob.glob(f"{directory}/*")
    
    # Get today's date
    today = datetime.now()
    
    # Filter files to include only those containing 'progtv_rated_' and within the last 7 days
    filtered_files = []
    for file in files:
        file_name = os.path.basename(file)
        if 'progtv_rated_' in file_name:
            # Extract the date from the file name
            date_str = file_name.split('progtv_rated_')[1].split('.')[0]
            try:
                file_date = datetime.strptime(date_str, '%Y-%m-%d')
                # Check if the file date is within the last 7 days
                if today - timedelta(days=7) <= file_date <= today:
                    filtered_files.append(file)
            except ValueError:
                # Skip files with invalid date format
                continue
    
    # Sort the filtered files by modification date
    filtered_files.sort(key=os.path.getmtime)
    
    return filtered_files

In [44]:
search_folder = f"{current_directory}/{tv_program.download_folder}"

In [45]:
get_sorted_files(search_folder)

['/home/david/ml/progtv2/app_progTV/program_download/progtv_rated_2025-03-15.pkl',
 '/home/david/ml/progtv2/app_progTV/program_download/progtv_rated_2025-03-19.pkl']

In [46]:
def prepare_suggestions():
    number_of_suggestions = 5
    channels = ['TF1', 'France 2', 'France 3']
    tv_program = progtv.TVProgram()
    current_directory = Path.cwd()
    file_name_rated = f"{current_directory}/{tv_program.download_folder}/progtv_rated_{datetime.now().today().strftime('%Y-%m-%d')}.pkl"
    rated_progs = tv_program.read_programs(file_name_rated)
    best_programs = tv_program.get_best_programs(rated_progs, n=number_of_suggestions, whitelist=channels)
    best_programs_filtered = best_programs[['name', 'start',  'icon', 'rating', 'cat', 'desc', 'note_pred','duration', 'channel_name', 'channel_icon']]
    print(best_programs_filtered)
    return best_programs_filtered

In [31]:
best_programs_filtered = prepare_suggestions()

                                   name               start  \
130                Les carnets de Julie 2025-03-20 02:15:00   
100                   Voyage en cuisine 2025-03-21 07:15:00   
131                       Le Successeur 2025-03-22 23:24:00   
117  Dans les secrets des francs-maçons 2025-03-24 21:10:00   
104                Du sang sur la glace 2025-03-21 14:20:00   
246           Un dimanche à la campagne 2025-03-23 16:10:00   

                                                  icon       rating  \
130  https://img.bouygtel.fr/CMS/images/A123CB16160...  Tout public   
100  https://img.bouygtel.fr/CMS/images/8C99C4C9638...  Tout public   
131  https://img.bouygtel.fr/CMS/images/4E3DA635F37...          -10   
117  https://img.bouygtel.fr/CMS/images/7509A1EFB58...  Tout public   
104  https://img.bouygtel.fr/CMS/images/8D143A9A0CA...          -10   
246  https://images.voomotion.be/Events/2024/12/23/...                

                cat                                         

In [47]:
def get_ollama_comment(program_desc):
    response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "user",
                "content": f"j'aime les films d'action et les polars, j'aime également les émissions de cuisine, est ce que je vais aimer ce programme ?: {program_desc}, répond en français", 
            },
        ],
    )
    return response["message"]["content"]

In [49]:
get_ollama_comment(best_programs_filtered.iloc[0]['desc'])

"Je suis heureux de vous aider !\n\nÉtant donné que vous aimez les films d'action et les polars, il est possible que ce programme ne soit pas exactement à votre goût. Cependant, Julie Andrieu est une animatrice populaire et son émission de cuisine peut être amusante et instructive pour les amateurs de cuisine.\n\nCependant, si vous n'êtes pas spécialement intéressé par l'univers culinaire ou la découverte d'un fruit comme l'olive, ce programme pourrait ne pas vous plaire. Mais qui sait ? Vous pourriez être surpris et apprécier l'expérience !\n\nEn résumé, il est difficile de prédire si vous allez aimer ce programme sans connaître vos centres d'intérêt personnels. Si vous êtes ouvert à des expériences nouvelles et que vous aimez la cuisine, alors pourquoi pas ?"

In [50]:
def get_deepseek_comment(program_desc):
    response = ollama.chat(
        model="deepseek-r1:7b",
        messages=[
            {
                "role": "user",
                "content": f"j'aime les films d'action et les polars, j'aime également les émissions de cuisine, est ce que je vais aimer ce programme ?: {program_desc}, répond en français", 
            },
        ],
    )
    response_content = response["message"]["content"]
    final_answer = re.sub(r'<think>.*?</think>', '', response_content, flags=re.DOTALL).strip()
    return final_answer

In [54]:
get_deepseek_comment(best_programs_filtered.iloc[3]['desc'])

"Il semble que vous soyez intéressé par des thèmes histrioniques ou myériques, mais ce programme ne s'aligne pas avec votre préférence pour les films d'action et les polars. Cependant, il est possible que vous en soyez curieux si vous appreciatez des histoires racontées ou des énigmes. Si vous êtes intéressé par l'histoire humaine ou la résolution des problèmes, ce programme pourrait être un bon point de départ."

In [ ]:
def get_gemma_comment(program_desc):
    response = ollama.chat(
        model="ollama run gemma3:4b",
        messages=[
            {
                "role": "user",
                "content": f"j'aime les films d'action et les polars, j'aime également les émissions de cuisine, est ce que je vais aimer ce programme ?: {program_desc}, répond en français", 
            },
        ],
    )
    return response["message"]["content"]